# Finding data of interest

The Delphi Epidata API includes numerous data streams -- medical claims,
cases and deaths, mobility, wastewater, and many others -- across different
geographic regions. Finding the one you want can be a challenge.

Data streams fall into three categories: **V5 sources**, **migrating
endpoints**, and **historical endpoints** (which include international sources
and private, authenticated endpoints).

In [ ]:
# Hidden cell (set in the metadata for this cell)
import pandas as pd

# Set common options and context
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 1000)

## V5 sources

The V5 API is the primary interface for active epidemiological surveillance
data.

### Online documentation and EpiPortal

The online docs list every source and signal on the [Delphi V5
API](https://cmu-delphi.github.io/delphi-epidata/api/v5_signals.html).

For interactive exploration, the [Delphi
EpiPortal](https://delphi.cmu.edu/epiportal/) lets you filter sources and
signals by disease, pathogen, geography, and date range, preview charts, and
copy query code.

### Exploring metadata with `epidata_meta()`

For V5 sources (queried with `epidata_snapshot()` and `epidata_archive()`),
`epidata_meta()` is the metadata lookup. Called with no arguments it lists
every active V5 source:

In [ ]:
from epidatpy import EpiDataContext, EpiRange

epidata = EpiDataContext(use_cache=True, cache_max_age_days=7)

meta = epidata.epidata_meta()
sorted(meta)

Called with `source=...`, it reports that source's signals, supported
geographic levels, and the available reference-time and report-time ranges:

In [ ]:
nssp_meta = epidata.epidata_meta(source="nssp")

print("signals:", nssp_meta["signals"])
print("geo_types:", nssp_meta["geo_types"])
print("reference_time_range:", nssp_meta["reference_time_range"])
print("report_time_range:", nssp_meta["report_time_range"])

You can also flatten the metadata into a table to search across all sources
with pandas:

In [ ]:
signals_df = pd.DataFrame(
    [
        {
            "source": src,
            "signal": signal,
            "geo_types": ", ".join(src_meta.get("geo_types", [])),
        }
        for src, src_meta in meta.items()
        for signal in src_meta.get("signals", [])
    ]
)

# Signals mentioning "flu"
signals_df[signals_df["signal"].str.contains("flu", case=False)]

### Example queries for V5 sources

`epidata_snapshot()` fetches data as of a given moment (latest by default);
`epidata_archive()` fetches the full revision history. See the [getting
started guide](getting_started.ipynb) for an introduction and the [versioned
data guide](versioned_data.ipynb) for versioning details.

Examples across several major V5 surveillance streams:

In [ ]:
# NSSP: influenza emergency department visits
epidata.epidata_snapshot(
    source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    geo_values="pa",
    reference_time=EpiRange("2024-10-01", "2024-10-15"),
).df()

In [ ]:
# POPHIVE: influenza emergency-department visit percentage
epidata.epidata_snapshot(
    source="pophive",
    signals="flu_pct_ed",
    geo_type="state",
    geo_values="pa",
    reference_time=EpiRange("2024-10-01", "2024-10-15"),
).df()

In [ ]:
# NWSS: wastewater SARS-CoV-2 concentration
epidata.epidata_snapshot(
    source="nwss",
    signals="covid_avg_conc",
    geo_type="sewershed",
    geo_values="128",
    reference_time=EpiRange("2024-12-01", "2024-12-15"),
).df()

In [ ]:
# Archive: revision history for a single reference date
epidata.epidata_archive(
    source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    geo_values="pa",
    reference_time="2024-12-07",
    report_time=EpiRange("2024-12-01", "2024-12-15"),
).df()

## Migrating endpoints

Datasets that originated in the legacy API -- `pub_covidcast()`,
`pub_covidcast_meta()`, `pub_fluview()`, `pub_fluview_clinical()`,
`pub_fluview_meta()`, `pub_flusurv()`, `pub_meta()` -- are transitioning to
V5. They still work for sources that have not moved yet.

### Exploring legacy COVIDcast sources with `CovidcastEpidata`

For datasets still queried through `pub_covidcast()`, `CovidcastEpidata`
describes every COVIDcast data source and signal from within the client. The
`source_df` property returns a DataFrame of all publicly accessible COVIDcast
data streams, mirroring the [COVIDcast signals
endpoint](https://cmu-delphi.github.io/delphi-epidata/api/covidcast_signals.html):

In [ ]:
from epidatpy import CovidcastEpidata

covidcast = CovidcastEpidata()
covidcast.source_df

`source_df` columns:

- `source` -- API-internal source name.
- `name` -- human-readable source name.
- `description` -- source description.
- `reference_signal`, `license`, `dua` -- reference signal, license, and Data
  Use Agreement link.
- `signals` -- signals available from this source.

`signal_df` describes each signal -- what time range it covers, when it was
last updated, and more:

In [ ]:
covidcast.signal_df

`signal_df` has one row per signal, with columns including `source`, `signal`,
`name`, `active`, `short_description`, `description`, `geo_types`, `time_type`,
`time_label`, `value_label`, `format`, `category`, `high_values_are`,
`is_smoothed`, `is_weighted`, `is_cumulative`, `has_stderr`, and
`has_sample_size`.

### Example legacy query

Legacy endpoints stay accessible while their sources transition (with a
deprecation warning):

In [ ]:
epidata.pub_covidcast(
    data_source="fb-survey",
    signals="smoothed_accept_covid_vaccine",
    geo_type="state",
    time_type="day",
    time_values=EpiRange(20201221, 20201225),
    geo_values="pa",
).df()

#### KCDC ILI

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/kcdc_ili.html>


In [ ]:
epidata.pub_kcdc_ili(regions="ROK", epiweeks=200436).df()

#### NIDSS Flu

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/nidss_flu.html>


In [ ]:
epidata.pub_nidss_flu(regions="taipei", epiweeks=EpiRange(200901, 201301)).df()

#### ILI Nearby Nowcast

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/nowcast.html>


In [ ]:
epidata.pub_nowcast(locations="ca", epiweeks=EpiRange(202201, 202319)).df()

### Dengue Endpoints

#### Delphi's Dengue Nowcast

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/dengue_nowcast.html>


In [ ]:
epidata.pub_dengue_nowcast(locations="pr", epiweeks=EpiRange(201401, 202301)).df()

#### NIDSS Dengue

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/nidss_dengue.html>


In [ ]:
epidata.pub_nidss_dengue(locations="taipei", epiweeks=EpiRange(200301, 201301)).df()

#### PAHO Dengue

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/paho_dengue.html>


In [ ]:
epidata.pub_paho_dengue(regions="ca", epiweeks=EpiRange(200201, 202319)).df()

### Other Endpoints

#### Wikipedia Access

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/wiki.html>


In [ ]:
epidata.pub_wiki(
  language="en",
  articles="influenza",
  time_type="week",
  time_values=EpiRange(202001, 202319)
).df()

### CAST API

CAST-API sources (e.g. `nssp`, `nwss`, `pophive`) are versioned differently from the classic `pub_covidcast` signals, via `epidata_snapshot()`, `epidata_archive()`, and `epidata_meta()`.

#### Auxiliary Data

`epidata_aux()` fetches time-varying auxiliary metadata attached to a cast source (e.g. NWSS sample-site descriptors). See the source's own API docs for its auxiliary key columns and allowed values, e.g. [NWSS](https://cmu-delphi.github.io/delphi-epidata/api/v5-signals/nwss.html).

Narrow with key filters -- an unfiltered pull returns the full report-time history for every matching row, which can be large.


In [ ]:
epidata.epidata_aux("nwss", sample_index="5886455", pcr_target="sars-cov-2").df()

`epidata_aux()` can also merge auxiliary data onto an already-fetched `epidata_snapshot()` or `epidata_archive()` result -- the source is recovered automatically. A snapshot for a broadly reporting source like `nwss` spans many sites, so it is left commented out here rather than run automatically; narrow it (e.g. by `geo_values`, or a smaller `reference_time` window) before merging in practice.


In [ ]:
# snapshot = epidata.epidata_snapshot(
#   source="nwss",
#   signals="covid_avg_conc",
#   geo_type="sewershed"
# ).df()
# epidata.epidata_aux(snapshot)

### Private methods

These require private access keys to use.

#### Google Health Trends

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/ght.html>


In [ ]:
# Requires valid API key
# epidata.pvt_ght(
#   auth="<YOUR_API_KEY>",
#   epiweeks=EpiRange(199301, 202304),
#   locations="ma",
#   query="how to get over the flu"
# ).df()

#### CDC

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/cdc.html>


In [ ]:
# epidata.pvt_cdc(auth="...", locations="ma", epiweeks=EpiRange(202003, 202304)).df()

#### Dengue Digital Surveillance Sensors

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/dengue_sensors.html>


In [ ]:
# epidata.pvt_dengue_sensors(
#   auth="...",
#   names="ght",
#   locations="ag",
#   epiweeks=EpiRange(201404, 202004)
# ).df()

#### NoroSTAT metadata

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/meta_norostat.html>


In [ ]:
# epidata.pvt_meta_norostat(auth="...").df()

#### NoroSTAT data

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/norostat.html>


In [ ]:
# epidata.pvt_norostat(auth="...", locations="1", epiweeks=201233).df()

#### Quidel Influenza testing

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/quidel.html>


In [ ]:
# epidata.pvt_quidel(auth="...", locations="hhs1", epiweeks=EpiRange(200301, 202105)).df()

#### Sensors

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/sensors.html>


In [ ]:
# epidata.pvt_sensors(
#   auth="...",
#   names="sar3",
#   locations="nat",
#   epiweeks=EpiRange(200301, 202105)
# ).df()

#### Twitter

# PAHO dengue digital surveillance sensors
epidata.pvt_dengue_sensors(
    auth="<SECRET>", names="ght", locations="ag", epiweeks=EpiRange(201404, 202004)
).df()
```